# Data compilation & cleaning — Lausanne residential mutations (monthly exports)

**Pipeline notebook 1.** Reads the monthly BDCH mutation exports (`YYYY-MM.csv`,
2015-01 → 2026-05), validates that they share a single schema, concatenates them
into one event-level master table, applies minimal privacy cleaning, and saves the
result for the downstream notebooks (trajectory reconstruction, household typology,
feature engineering, modelling).

This notebook **supersedes** the earlier semi-annual pipeline: the monthly re-exports
use a different, homogeneous schema (67 columns, comma-separated, UTF-8 with accents).
The old variable catalogue no longer maps one-to-one — key renames include
`idmen→NUMERO_MENAGE`, `typmen_c→TYPE_MENAGE`, `ETACIVIL_c→ETAT_CIVIL`,
`SEXE_c→CODE_SEXE`. The new files add household size (`PERSMEN`), a four-way
household type, and *before-mutation* household state (`MUTATION_*`), but drop the
family-nucleus and head-of-household variables.

**Scope of this notebook:** compilation + structural cleaning only. Date typing,
value normalisation and derived variables (household typology, departure target) are
handled in later notebooks, as each requires documented methodological choices.


## 0. Configuration

All settings live here: paths, CSV format, the column-role mapping for the new
schema, and identifier handling. The monthly files are **comma-separated** and
**UTF-8 with accented characters**; UTF-8 is tried first, with `cp1252`/`latin-1`
as safe fallbacks. Everything is read as text (`dtype=str`) to preserve identifiers
and leading zeros — typing happens later, deliberately.


In [9]:
from pathlib import Path
import re
import pandas as pd

# ---- Paths -------------------------------------------------------------
# Monthly CSVs live under DATA_DIR (optionally in per-year subfolders).
DATA_DIR   = Path("/mnt/o/09_STATISTIQUES/09_04_Explorations/MR/CAS ADS/Final project/data")
OUT_DIR    = DATA_DIR                              # where the master file is written
MASTER_OUT = OUT_DIR / "masterfile_brut.parquet"

# ---- CSV format (homogeneous across the monthly corpus) ----------------
CSV_SEP   = ","                                    # monthly files are comma-separated
ENCODINGS = ("utf-8-sig", "cp1252", "latin-1")     # accents: UTF-8 first, safe fallbacks

# ---- File-name pattern: the stem must be EXACTLY "YYYY-MM" --------------
# This excludes the legacy semi-annual files (e.g. BDCHMutations2015-01-01-...).
MONTH_RE = re.compile(r"^(20\d{2})-(0[1-9]|1[0-2])$")

# ---- Identifier handling (privacy) -------------------------------------
ID_KEEP   = "NOREFCH"          # stable individual key -> kept in clear text as id_projet
ID_RENAME = "id_projet"
ID_DROP   = ["NOAVS", "IDHAB"] # direct / sensitive identifiers removed at ingestion

# ---- Column roles (NEW monthly schema) ---------------------------------
# Documented so downstream notebooks reference columns by role, not by guess.
DATE_COLS = ["DATE_EFFECTIVE", "MUTATION_DATE", "DATNAIS", "DDECES", "DETATCIVIL",
             "DATEENTCH", "DATARR", "DATDEP", "DATDEM", "XFERESID"]
PERSON_COLS    = ["DATNAIS", "CODE_SEXE", "ETAT_CIVIL", "NATION"]
MUTATION_COLS  = ["CODE_MUTATION", "TYPE_ADRESSE"]
HOUSEHOLD_COLS = ["NUMERO_MENAGE", "RM_ID_MENAGE", "TYPE_MENAGE", "PERSMEN",
                  "MENAGE_EWID", "EGID", "NOMBRE_PIECE", "SURFACE"]
HOUSEHOLD_PREV = ["MUTATION_NUMERO_MENAGE", "MUTATION_TYPEMENAGE", "MUTATION_PERSMEN"]

assert DATA_DIR.exists(), f"DATA_DIR not found: {DATA_DIR}"
print("Data directory:", DATA_DIR)

Data directory: /mnt/o/09_STATISTIQUES/09_04_Explorations/MR/CAS ADS/Final project/data


## 1. File discovery

Recursively collect every `YYYY-MM.csv` under `DATA_DIR`, ignoring any legacy
semi-annual file (whose stem is not exactly `YYYY-MM`) and anything in an `anciens`
folder. The period (e.g. `2015-03`) is parsed from the file name and will serve as
provenance and as the temporal ordering key.


In [10]:
def period_from_name(stem: str):
    m = MONTH_RE.match(stem)
    return f"{m.group(1)}-{m.group(2)}" if m else None

files = sorted(
    p for p in DATA_DIR.rglob("*.csv")
    if MONTH_RE.match(p.stem) and "anciens" not in p.parts
)
assert files, f"No monthly file (YYYY-MM.csv) found under {DATA_DIR}"

periods = [period_from_name(p.stem) for p in files]
print(f"{len(files)} monthly files: {periods[0]} ... {periods[-1]}")

# Flag duplicated periods (same month imported twice / present in two folders).
dups = sorted({p for p in periods if periods.count(p) > 1})
print("Duplicated periods:", dups or "none")

137 monthly files: 2015-01 ... 2026-05
Duplicated periods: none


## 2. Robust single-file reader

One function reads a comma-separated monthly file as text, trying the encodings in
order so accented characters decode correctly. A residual BOM and any trailing
`Unnamed` columns are stripped. A quick probe on the first file confirms the format
and prints the column inventory once.


In [11]:
def read_monthly(path: Path) -> pd.DataFrame:
    """Read one comma-separated monthly file as text, trying ENCODINGS in order."""
    last_err = None
    for enc in ENCODINGS:
        try:
            df = pd.read_csv(path, sep=CSV_SEP, encoding=enc, dtype=str,
                             keep_default_na=True,
                             na_values=["", " ", "NULL", "null", "NA"],
                             low_memory=False)
            df.columns = [c.strip().lstrip("\ufeff") for c in df.columns]
            df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")])
            return df
        except UnicodeDecodeError as e:
            last_err = e
    raise RuntimeError(f"Could not decode {path.name}: {last_err}")

probe = read_monthly(files[0])
print(f"{files[0].name}: {probe.shape[0]} rows x {probe.shape[1]} columns\n")
print(sorted(probe.columns))

2015-01.csv: 5119 rows x 67 columns

['AUTRECOM', 'AUTRPAYS', 'CODETA_C', 'CODE_MUTATION', 'CODE_SEXE', 'CODRUE', 'COMDEST', 'COMDEST_C', 'COMNAIS', 'COMPROV', 'COMPROV_C', 'DATARR', 'DATDEM', 'DATDEP', 'DATEENTCH', 'DATE_EFFECTIVE', 'DATNAIS', 'DDECES', 'DETATCIVIL', 'EGID', 'ETAT_CIVIL', 'IDHAB', 'LIBADRES_C', 'LOCALITE_C', 'MENAGE_EWID', 'MUTATION_CODRUE', 'MUTATION_COMPROV', 'MUTATION_COMPROV_C', 'MUTATION_DATE', 'MUTATION_EGID', 'MUTATION_EWID', 'MUTATION_LIBADRES_C', 'MUTATION_LOCALITE_C', 'MUTATION_NOCOM', 'MUTATION_NOPOST_C', 'MUTATION_NUMERO_MENAGE', 'MUTATION_PAYSPROVC', 'MUTATION_PAYSPROVN', 'MUTATION_PERSMEN', 'MUTATION_PIECES', 'MUTATION_SURFACE', 'MUTATION_TYPADRES_C', 'MUTATION_TYPEMENAGE', 'NATION', 'NOAVS', 'NOCOM', 'NOIMM_C', 'NOMBRE_PIECE', 'NOPOST_C', 'NOREFCH', 'NUMERO_ENTREE', 'NUMERO_MENAGE', 'PAYSDEST', 'PAYSDEST_C', 'PAYSNAIS', 'PAYSPROVC', 'PAYSPROVN', 'PERMIS_C', 'PERSMEN', 'RM_ID_MENAGE', 'SURFACE', 'TYPADRES_C', 'TYPAD_C', 'TYPERM_C', 'TYPE_ADRESSE', 'TYPE_

## 3. Schema audit

Before concatenating, confirm that all 137 files really share the same columns.
This is a **header-only** check (`nrows=0`) so it is fast and does not re-read the
data. Any divergent file is reported with the symmetric difference against the
reference schema — a silent misalignment in `concat` would otherwise be hard to spot.


In [12]:
def header_cols(path: Path):
    for enc in ENCODINGS:
        try:
            h = pd.read_csv(path, sep=CSV_SEP, encoding=enc, nrows=0)
            return [c.strip().lstrip("\ufeff") for c in h.columns
                    if not c.startswith("Unnamed")]
        except UnicodeDecodeError:
            continue
    raise RuntimeError(f"Could not decode header of {path.name}")

schemas   = {p.name: header_cols(p) for p in files}
ref       = schemas[files[0].name]
divergent = {n: sorted(set(c) ^ set(ref)) for n, c in schemas.items() if set(c) != set(ref)}

print(f"Reference schema: {len(ref)} columns")
if divergent:
    print(f"WARNING - {len(divergent)} file(s) differ from the reference:")
    for n, d in list(divergent.items())[:10]:
        print(f"  {n}: {d}")
else:
    print("All files share the same columns.")

Reference schema: 67 columns
All files share the same columns.


## 4. Assemble the raw master table

Read every file, append provenance (`source_file`, `periode`, `annee`), and
concatenate. An assertion guarantees no rows are lost or duplicated. No
`cohorte_schema` flag is needed: unlike the old corpus, the monthly exports have a
single uniform schema across the whole period.


In [ ]:
frames, manifest = [], []
for p, period in zip(files, periods):
    df = read_monthly(p)
    df["source_file"] = p.name
    df["periode"]     = period
    df["annee"]       = int(period[:4])
    frames.append(df)
    manifest.append({"periode": period, "rows": len(df), "cols": df.shape[1] - 3})
    if len(df) == 0:
        print(f"WARNING - empty file: {p.name}")

manifest = pd.DataFrame(manifest).sort_values("periode").reset_index(drop=True)
master   = pd.concat(frames, ignore_index=True, sort=False)

assert len(master) == manifest["rows"].sum(), "Row count mismatch after concat!"

# --- Corpus completeness guard: a monthly file can exist on disk yet be
# --- empty or mislabeled; fail loudly rather than warn ----------------------
empty = manifest.loc[manifest["rows"] == 0, "periode"].tolist()
assert not empty, f"Empty monthly file(s) -- re-extract before proceeding: {empty}"

expected_m = pd.period_range(manifest["periode"].min(), manifest["periode"].max(),
                             freq="M").astype(str)
missing_m = sorted(set(expected_m) - set(manifest["periode"]))
assert not missing_m, f"Missing month(s) in the corpus: {missing_m}"
min_rows = int(manifest["rows"].min())
print(f"Corpus completeness: every month present and non-empty "
      f"(smallest month: {min_rows:,} rows).")
print(f"Master table: {len(master):,} rows x {master.shape[1]} columns")
print(f"Coverage: {manifest['periode'].min()} -> {manifest['periode'].max()}")
print(f"Data columns identical everywhere: "
      f"{manifest['cols'].nunique() == 1} ({manifest['cols'].iloc[0]} columns)")

## 5. Validation & completeness

Three structural checks, all data-driven (nothing assumed):

- **Key column presence** — confirm the analytically important variables are there
  under their new names.
- **Fully empty columns** — reported for review (not auto-dropped: an empty column
  may still be meaningful metadata).
- **Completeness over time** — fill rate of the household variables by year, to
  verify that `PERSMEN`, `TYPE_MENAGE`, `NUMERO_MENAGE` and their `MUTATION_*`
  counterparts are populated across the whole period and not only in recent years.


In [14]:
key_cols = [ID_KEEP] + PERSON_COLS + MUTATION_COLS + HOUSEHOLD_COLS + HOUSEHOLD_PREV
print("Key column presence:")
for c in dict.fromkeys(key_cols):                 # de-duplicate, keep order
    print(f"  {c:24} {'present' if c in master.columns else 'MISSING'}")

empty_cols = [c for c in master.columns if master[c].notna().sum() == 0]
print("\nFully empty columns:", empty_cols or "none")

watch = [c for c in HOUSEHOLD_COLS + HOUSEHOLD_PREV if c in master.columns]
fill_by_year = master.groupby("annee")[watch].agg(lambda s: s.notna().mean()).round(2)
print("\nFill rate of household variables by year:")
print(fill_by_year.to_string())

Key column presence:
  NOREFCH                  present
  DATNAIS                  present
  CODE_SEXE                present
  ETAT_CIVIL               present
  NATION                   present
  CODE_MUTATION            present
  TYPE_ADRESSE             present
  NUMERO_MENAGE            present
  RM_ID_MENAGE             present
  TYPE_MENAGE              present
  PERSMEN                  present
  MENAGE_EWID              present
  EGID                     present
  NOMBRE_PIECE             present
  SURFACE                  present
  MUTATION_NUMERO_MENAGE   present
  MUTATION_TYPEMENAGE      present
  MUTATION_PERSMEN         present

Fully empty columns: none

Fill rate of household variables by year:
       NUMERO_MENAGE  RM_ID_MENAGE  TYPE_MENAGE  PERSMEN  MENAGE_EWID  EGID  NOMBRE_PIECE  SURFACE  MUTATION_NUMERO_MENAGE  MUTATION_TYPEMENAGE  MUTATION_PERSMEN
annee                                                                                                                

## 6. Minimal privacy cleaning

Only structural, decision-free operations here:

- Drop direct/sensitive identifiers (`NOAVS`, `IDHAB`); the AVS number especially is
  a national identifier that should not travel through the analysis.
- Rename the linkage key `NOREFCH → id_projet`, kept in clear text (no hashing), as
  decided for this project.

Date parsing and value normalisation are intentionally left to the next notebook.


In [15]:
clean = master.copy()

to_drop = [c for c in ID_DROP if c in clean.columns]
clean = clean.drop(columns=to_drop)
print("Dropped identifiers:", to_drop or "none")

if ID_KEEP in clean.columns:
    clean = clean.rename(columns={ID_KEEP: ID_RENAME})
    print(f"Renamed {ID_KEEP} -> {ID_RENAME}")

print(f"\nCleaned master: {len(clean):,} rows x {clean.shape[1]} columns")

Dropped identifiers: ['NOAVS', 'IDHAB']
Renamed NOREFCH -> id_projet

Cleaned master: 862,304 rows x 68 columns


## 7. Save the master table

Saved as Parquet when an engine is available (types and values preserved on
re-read). If no Parquet engine works — e.g. `pyarrow` missing or a version conflict
on Python 3.14 — the code falls back to compressed CSV automatically. With CSV,
re-read using `dtype=str` to keep identifiers intact.


In [16]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
try:
    clean.to_parquet(MASTER_OUT, index=False)
    saved = MASTER_OUT
except Exception as e:                            # missing engine OR pyarrow/pandas conflict
    saved = MASTER_OUT.with_suffix(".csv.gz")
    clean.to_csv(saved, index=False, compression="gzip")
    print(f"Parquet unavailable ({type(e).__name__}) -> CSV gzip. Re-read with dtype=str.")
print(f"Saved: {saved}  ({saved.stat().st_size/1e6:.1f} MB)")

Saved: /mnt/o/09_STATISTIQUES/09_04_Explorations/MR/CAS ADS/Final project/data/masterfile_brut.parquet  (59.0 MB)


## 8. Notes & next steps

- **Date formats — verify before typing.** In the old corpus, `MUTATION_DATE` /
  `DATE_EFFECTIVE` were ISO timestamps while personal dates (`DATNAIS`, `DATARR`,
  `DATDEP`, …) were Excel serial numbers. Do **not** assume this still holds in the
  re-exports — inspect a sample of each `DATE_COLS` column first, then choose the
  parser per column in the next notebook.
- **Household typology.** With `PERSMEN` (size), the four-way `TYPE_MENAGE`,
  `NUMERO_MENAGE`, and the `MUTATION_*` before-state available for every year, the
  typology can be built on the full 2015–2026 span. Composition (couple / single
  parent / non-family) is still **inferred** from `ETAT_CIVIL` + `DATNAIS` within
  `NUMERO_MENAGE`, since there is no family-nucleus or head-of-household variable.
- **Downstream order:** (next) date typing & quality audit → deduplication and
  household state reconstruction at a reference date → household typology → feature
  engineering for the binary departure target → EDA.
